# 04 · Results: does VA-HDW help, and which model is final?

1. Scores of every run, as mean ± SD over the 5 outer test folds, on **all test cells** (measured + filled, the main result) and on **measured cells only**.
2. **Does VA-HDW help?** (rule decided before the results) For each model, the fold-by-fold difference in Macro-F1, XYZ+VA-HDW minus XYZ, on the same test boreholes. VA-HDW **helps** only if the mean difference is larger than one standard error of the differences; **hurts** if below minus one SE; otherwise **no clear difference**.
3. **The final model** (rule decided before the results). Every model on every feature set is a candidate:
   * eligible: mean Clay and Sand recall ≥ 0.50, and no fold below 0.25;
   * best = highest mean Macro-F1;
   * among the eligible candidates within one standard error of the best: lowest fold SD of Macro-F1, then lowest Brier score.
4. Scores by distance to the nearest training borehole; block-size sensitivity and purge.

**Run time:** under 1 minute.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])
preds = pd.read_csv(H.out_dir(S, "03_spatial_cv", private=True) / "outer_predictions.csv")
OUT = H.out_dir(S, "04_results")
folds_tab = H.fold_scores(preds)
summ = H.summary(folds_tab)
folds_tab.to_csv(OUT / "scores_by_fold.csv", index=False)
summ.to_csv(OUT / "scores_mean_sd.csv", index=False)
show = ["model", "features", "macro_f1_mean", "macro_f1_sd", "accuracy_mean", "balanced_accuracy_mean",
        "recall_clay_mean", "recall_sand_mean", "roc_auc_mean", "brier_mean", "n_folds"]
main = summ[(summ.multiple == 1) & (summ.purge_x == 0)]
for c in ("all", "measured"):
    print(f"Main block size ({main.block_m.iloc[0]:.0f} m), {c} test cells:")
    display(main[main.cells == c][show].sort_values("macro_f1_mean", ascending=False).round(3))

## 1. Does VA-HDW help? (paired test, same folds)

In [ ]:
gain = pd.concat([H.paired_gain(folds_tab, c).assign(cells=c) for c in ("all", "measured")], ignore_index=True)
gain.to_csv(OUT / "vahdw_gain.csv", index=False)
display(gain.round(3))

## 2. The final model (rule decided before the results)

In [ ]:
sel = H.select_model(summ, "all")
display(sel["table"][["candidate", "eligible", "macro_f1_mean", "macro_f1_sd", "brier_mean"]]
        .sort_values("macro_f1_mean", ascending=False).round(3))
print("Highest mean Macro-F1:", sel.get("highest_mean"), "| one SE:", round(sel.get("one_se", np.nan), 4))
print("Within one SE:", sel.get("within_1se"))
print("FINAL MODEL:", sel["selected"], "|", sel.get("features"))
sel_m = H.select_model(summ, "measured")
print("(for comparison, the same rule on measured cells only would choose:", sel_m["selected"], "|", sel_m.get("features"), ")")
_ = (OUT / "selection.json").write_text(json.dumps({k: v for k, v in sel.items() if k != "table"}, indent=1, default=str))

## 3. Model comparison figure (quick look; the thesis figures are in notebook 09)

In [ ]:
m = main[main.cells == "all"]
models = [x for x in list(S["cv"]["models"]) + list(S["cv"]["vahdw_only_models"]) + list(S["cv"]["xyz_only_models"])
          if x in set(m.model)]
fig, ax = plt.subplots(figsize=(8, 4))
for j, (f, col) in enumerate((("XYZ", "#9E9E9E"), ("XYZ_VAHDW", "#17365C"))):
    t = m[m.features == f].set_index("model").reindex(models)
    x = np.arange(len(models)) + (j - 0.5) * 0.38
    ax.bar(x, t.macro_f1_mean, width=0.36, yerr=t.macro_f1_sd, color=col, capsize=3, label=f.replace("_", " + "))
ax.set_xticks(np.arange(len(models)), models)
ax.set_ylabel("Macro-F1 (mean ± SD, outer folds)"); ax.set_ylim(0, 1); ax.legend(frameon=False)
ax.set_title(f"Spatial CV, {m.block_m.iloc[0]:.0f} m blocks; final: {sel['selected']} | {sel.get('features')}", fontsize=9)
fig.savefig(OUT / "fig_model_comparison.png", dpi=200, bbox_inches="tight")

## 4. By distance to the nearest training borehole
Does VA-HDW help close to boreholes, far from them, or both?

In [ ]:
bands = H.distance_bands(preds[(preds.multiple == 1) & (preds.purge_x == 0)])
bands.to_csv(OUT / "scores_by_distance.csv", index=False)
t = bands[bands.cells == "all"].pivot_table(index=["model", "features"], columns="band", values="macro_f1", observed=True)
n = bands[(bands.cells == "all")].groupby("band", observed=True).n.max()
print("Macro-F1 by distance band (all cells); test cells per band:", n.to_dict())
display(t.round(3))

## 5. Block-size sensitivity and purge

In [ ]:
sens = summ[(summ.purge_x == 0) & (summ.cells == "all") & summ.model.isin(S["cv"]["sensitivity_models"])]
display(sens.pivot_table(index=["model", "features"], columns="block_m", values="macro_f1_mean").round(3))
pur = summ[(summ.multiple == 1) & (summ.cells == "all") & summ.model.isin(S["cv"]["purge_models"])]
display(pur.pivot_table(index=["model", "features"], columns="purge_m", values="macro_f1_mean").round(3))
for x in sorted(set(summ.multiple)):
    for p in sorted(set(summ.purge_x)):
        g = H.paired_gain(folds_tab, "all", x, p)
        if len(g) and (x != 1 or p != 0):
            print(f"VA-HDW gain, block x{x}, purge x{p}:")
            display(g.round(3))
print(f"Finished in {time.time() - T0:.0f} s. Next: notebook 05.")